# 02 — Consume Kafka events into the bronze layer

Pulls messages off every topic and lands them as partitioned Parquet (a columnar
file format good for analytics) files under `data/bronze/<topic>/dt=<event
date>/*.parquet` — raw/unprocessed data that a future `warehouse/` project (dbt
(a SQL transformation tool) + DuckDB (an embedded analytics database)) will
consume.

`consume_batch()` keeps one long-lived Kafka consumer alive across calls within
this kernel session, so its committed offset naturally continues from the last
call. **Re-running the same cell is the "follow" mechanism** — no infinite loop
needed.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from consumers.bronze_writer import consume_batch

## Smoke test — consume a small batch

Always verify with a small `limit` first, same as the producer side.

In [ ]:
consume_batch(limit=20)

Check `streaming/data/bronze/` now — you should see one `dt=...` partition per
topic that had messages, each holding a `.parquet` file.

## Follow mode — re-run this cell while the producer paces out messages

With no `limit`, this call keeps reading until nothing new shows up for
`idle_timeout_seconds`, then returns. While `01_replay.ipynb`'s paced
`publish_next(events, limit=30, interval_seconds=3)` cell is running (in another
notebook/kernel), re-run this cell every so often and watch new Parquet files
accumulate under `data/bronze/`.

In [ ]:
consume_batch(idle_timeout_seconds=5)

## Drain the rest

Once the producer side has published everything, run this with a longer idle
timeout to mop up anything still in flight.

In [ ]:
consume_batch(idle_timeout_seconds=10)